# 🧬 DermRL — Experiment 1: DQN (Value-Based)
### SkinConditionEnv | Deep Q-Network | 10 Hyperparameter Experiments

DQN is a **value-based** method. It learns a Q-function Q(s,a) — the expected return for taking action *a* in state *s*. The policy is implicit: always pick the action with the highest Q-value (ε-greedy during training).

**Key hyperparameters explored:**
- Learning rate, buffer size, batch size, gamma (discount)
- Epsilon schedule (exploration), target network update frequency
- Network architecture depth

---
*Runtime: T4 GPU recommended · ~30–40 min for all 10 experiments*

## 1 · Install & Imports

In [1]:
%%capture
!pip install stable-baselines3[extra] gymnasium torch matplotlib pandas seaborn

In [3]:
import os, sys, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import EvalCallback

PROJECT_ROOT = '/content/skin_rl_project'
os.makedirs(f'{PROJECT_ROOT}/environment', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/models/dqn', exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

ModuleNotFoundError: No module named 'torch'

## 2 · Write Environment

In [ ]:
# Paste the full content of environment/custom_env.py below
ENV_CODE = '''
# ← PASTE environment/custom_env.py content here
'''
with open(f'{PROJECT_ROOT}/environment/__init__.py', 'w') as f: f.write('')
with open(f'{PROJECT_ROOT}/environment/custom_env.py', 'w') as f: f.write(ENV_CODE)
print('Environment written.')

In [ ]:
from environment.custom_env import SkinConditionEnv
env = SkinConditionEnv(render_mode='none')
obs, info = env.reset()
print('Obs shape:', obs.shape, '| Actions:', env.action_space.n)
print('Initial severity:', f"{info['severity']:.1%}")
env.close()
print('✅ Environment OK')

## 3 · Experiment Configurations

The 10 experiments systematically vary:
- **LR**: tests sensitivity to step size (too high → divergence, too low → slow convergence)
- **Buffer**: small buffers cause correlated updates; large buffers improve stability
- **Batch size**: larger = more stable gradients but slower per-step
- **Gamma**: how far-sighted the agent is (low γ = myopic)
- **Epsilon schedule**: controls exploration/exploitation tradeoff
- **Target update**: how often the stable Q-target is refreshed
- **Net architecture**: capacity of the function approximator

In [ ]:
EXPERIMENTS = [
    {
        "id": 1,
        "name": "Baseline",
        "desc": "Standard DQN reference point. Moderate LR, medium buffer, default epsilon decay.",
        "config": dict(learning_rate=1e-3, buffer_size=50_000, batch_size=64, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 2,
        "name": "High LR",
        "desc": "LR=5e-3: faster early learning but risks instability and overshooting Q-values.",
        "config": dict(learning_rate=5e-3, buffer_size=50_000, batch_size=64, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 3,
        "name": "Low LR",
        "desc": "LR=1e-4: slow but stable convergence. Good for fine-tuning later stages.",
        "config": dict(learning_rate=1e-4, buffer_size=50_000, batch_size=64, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 4,
        "name": "Large Buffer",
        "desc": "Buffer=200k: more diverse replay, less correlated updates, more stable training.",
        "config": dict(learning_rate=1e-3, buffer_size=200_000, batch_size=64, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 5,
        "name": "Small Buffer",
        "desc": "Buffer=5k: fast turnover of experiences but high correlation. Expect instability.",
        "config": dict(learning_rate=1e-3, buffer_size=5_000, batch_size=64, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 6,
        "name": "Low Gamma",
        "desc": "γ=0.85: myopic agent. Prioritises immediate reward, ignores long-term skin clearing.",
        "config": dict(learning_rate=1e-3, buffer_size=50_000, batch_size=64, gamma=0.85,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 7,
        "name": "High Gamma",
        "desc": "γ=0.999: far-sighted agent. Plans over full 90-day horizon. Risk: slow credit assignment.",
        "config": dict(learning_rate=1e-3, buffer_size=50_000, batch_size=64, gamma=0.999,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 8,
        "name": "More Exploration",
        "desc": "ε-final=0.20, fraction=0.40: keeps exploring longer. Useful in stochastic environments.",
        "config": dict(learning_rate=1e-3, buffer_size=50_000, batch_size=64, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.40, exploration_final_eps=0.20,
                       policy_kwargs=dict(net_arch=[64, 64]))
    },
    {
        "id": 9,
        "name": "Deeper Network",
        "desc": "3 hidden layers [256,256,256]: higher capacity, can model complex Q-landscapes.",
        "config": dict(learning_rate=1e-3, buffer_size=50_000, batch_size=128, gamma=0.99,
                       train_freq=4, gradient_steps=1, target_update_interval=500,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[256, 256, 256]))
    },
    {
        "id": 10,
        "name": "Frequent Target Update + Large Batch",
        "desc": "target_update=100, batch=256: very stable Q-targets with high-quality gradient estimates.",
        "config": dict(learning_rate=1e-3, buffer_size=100_000, batch_size=256, gamma=0.99,
                       train_freq=4, gradient_steps=2, target_update_interval=100,
                       exploration_fraction=0.20, exploration_final_eps=0.05,
                       policy_kwargs=dict(net_arch=[128, 128]))
    },
]
print(f'Loaded {len(EXPERIMENTS)} experiment configurations.')
for e in EXPERIMENTS:
    print(f"  Exp {e['id']:2d}: {e['name']}")

## 4 · Training Loop

In [ ]:
TOTAL_STEPS = 150_000
results = []

for exp in EXPERIMENTS:
    print(f"\n{'='*60}")
    print(f"Exp {exp['id']:2d}/10: {exp['name']}")
    print(f"  → {exp['desc']}")
    print(f"{'='*60}")

    save_dir = f"{PROJECT_ROOT}/models/dqn/exp_{exp['id']:02d}"
    os.makedirs(save_dir, exist_ok=True)

    train_env = Monitor(SkinConditionEnv(render_mode='none'))
    eval_env  = Monitor(SkinConditionEnv(render_mode='none', seed=42))

    eval_cb = EvalCallback(eval_env, best_model_save_path=save_dir,
                           log_path=save_dir, eval_freq=10_000,
                           n_eval_episodes=10, deterministic=True, verbose=0)

    model = DQN('MlpPolicy', train_env, verbose=0, device=DEVICE,
                learning_starts=1000, **exp['config'])

    t0 = time.time()
    model.learn(total_timesteps=TOTAL_STEPS, callback=eval_cb, progress_bar=True)
    elapsed = time.time() - t0

    # Final evaluation
    eval_env2 = Monitor(SkinConditionEnv(render_mode='none', seed=99))
    mean_r, std_r = evaluate_policy(model, eval_env2, n_eval_episodes=20, deterministic=True)

    # Success rate
    successes = 0
    for _ in range(50):
        obs, _ = eval_env2.reset()
        done = False
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, _, term, trunc, info = eval_env2.step(int(action))
            done = term or trunc
        if info.get('severity', 1.0) < 0.35:
            successes += 1

    result = {
        'exp_id':       exp['id'],
        'name':         exp['name'],
        'mean_reward':  round(mean_r, 2),
        'std_reward':   round(std_r, 2),
        'success_rate': round(successes / 50 * 100, 1),
        'train_time_s': round(elapsed, 1),
        'lr':           exp['config']['learning_rate'],
        'buffer':       exp['config']['buffer_size'],
        'batch':        exp['config']['batch_size'],
        'gamma':        exp['config']['gamma'],
        'eps_final':    exp['config']['exploration_final_eps'],
        'target_upd':   exp['config']['target_update_interval'],
        'net_arch':     str(exp['config']['policy_kwargs']['net_arch']),
    }
    results.append(result)
    print(f"  ✅ Mean reward: {mean_r:.2f} ± {std_r:.2f} | Success: {successes}/50 | Time: {elapsed/60:.1f}min")
    train_env.close(); eval_env.close(); eval_env2.close()

print('\n✅ All DQN experiments complete!')

## 5 · Results Table

In [ ]:
df = pd.DataFrame(results)
df = df.sort_values('mean_reward', ascending=False).reset_index(drop=True)

display_cols = ['exp_id','name','lr','buffer','batch','gamma','eps_final',
                'target_upd','net_arch','mean_reward','std_reward','success_rate','train_time_s']
df_display = df[display_cols].copy()
df_display.columns = ['#','Experiment','LR','Buffer','Batch','γ','ε-final',
                       'Target Upd','Net Arch','Mean R','Std R','Success %','Time(s)']

print('\n📊 DQN Hyperparameter Experiment Results (sorted by Mean Reward)')
print('='*120)
print(df_display.to_string(index=False))
print('='*120)

# Save CSV
df.to_csv(f'{PROJECT_ROOT}/models/dqn/dqn_results.csv', index=False)
print('\nResults saved to dqn_results.csv')

## 6 · Hyperparameter Analysis Plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('#0D1117')
plt.suptitle('DQN Hyperparameter Analysis — SkinConditionEnv', 
             fontsize=16, color='white', y=1.01, fontweight='bold')

plot_cfg = dict(facecolor='#0D1117')
label_cfg = dict(color='white', fontsize=10)

# 1. Mean reward per experiment
ax = axes[0,0]; ax.set_facecolor('#161B22')
colors = ['#2EA043' if r > 0 else '#CF222E' for r in df['mean_reward']]
bars = ax.bar(df['name'], df['mean_reward'], color=colors, edgecolor='#30363D')
ax.set_xticklabels(df['name'], rotation=45, ha='right', color='white', fontsize=8)
ax.set_ylabel('Mean Reward', **label_cfg); ax.set_title('Mean Reward by Experiment', color='white', fontsize=11)
ax.tick_params(colors='white'); [s.set_edgecolor('#30363D') for s in ax.spines.values()]
ax.errorbar(df['name'], df['mean_reward'], yerr=df['std_reward'], fmt='none', color='white', capsize=4)

# 2. LR vs Mean Reward
ax = axes[0,1]; ax.set_facecolor('#161B22')
lr_group = df.groupby('lr')['mean_reward'].agg(['mean','std']).reset_index()
ax.errorbar(lr_group['lr'], lr_group['mean'], yerr=lr_group['std'],
            fmt='o-', color='#58A6FF', linewidth=2, markersize=8, capsize=5)
ax.set_xscale('log'); ax.set_xlabel('Learning Rate (log scale)', **label_cfg)
ax.set_ylabel('Mean Reward', **label_cfg); ax.set_title('LR vs Mean Reward', color='white', fontsize=11)
ax.tick_params(colors='white'); [s.set_edgecolor('#30363D') for s in ax.spines.values()]

# 3. Buffer Size vs Mean Reward
ax = axes[0,2]; ax.set_facecolor('#161B22')
buf_group = df.groupby('buffer')['mean_reward'].mean().reset_index()
ax.bar(buf_group['buffer'].astype(str), buf_group['mean_reward'], color='#BC8EFF', edgecolor='#30363D')
ax.set_xlabel('Buffer Size', **label_cfg); ax.set_ylabel('Mean Reward', **label_cfg)
ax.set_title('Buffer Size vs Mean Reward', color='white', fontsize=11)
ax.tick_params(colors='white'); [s.set_edgecolor('#30363D') for s in ax.spines.values()]

# 4. Gamma vs Success Rate
ax = axes[1,0]; ax.set_facecolor('#161B22')
g_group = df.groupby('gamma')['success_rate'].mean().reset_index()
ax.bar(g_group['gamma'].astype(str), g_group['success_rate'], color='#F0883E', edgecolor='#30363D')
ax.set_xlabel('Discount Factor γ', **label_cfg); ax.set_ylabel('Success Rate (%)', **label_cfg)
ax.set_title('Gamma vs Success Rate', color='white', fontsize=11)
ax.tick_params(colors='white'); [s.set_edgecolor('#30363D') for s in ax.spines.values()]

# 5. Success Rate per Experiment
ax = axes[1,1]; ax.set_facecolor('#161B22')
s_colors = ['#2EA043' if s >= 50 else '#CF222E' for s in df['success_rate']]
ax.barh(df['name'], df['success_rate'], color=s_colors, edgecolor='#30363D')
ax.axvline(50, color='white', ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Success Rate (%)', **label_cfg); ax.set_title('Success Rate by Experiment', color='white', fontsize=11)
ax.tick_params(colors='white'); [s.set_edgecolor('#30363D') for s in ax.spines.values()]

# 6. Reward vs Std (stability)
ax = axes[1,2]; ax.set_facecolor('#161B22')
scatter = ax.scatter(df['std_reward'], df['mean_reward'], 
                      c=df['success_rate'], cmap='RdYlGn', s=120, edgecolors='white', lw=0.5)
for _, row in df.iterrows():
    ax.annotate(f"Exp{row['exp_id']}", (row['std_reward'], row['mean_reward']),
                textcoords='offset points', xytext=(5,3), color='white', fontsize=7)
cbar = plt.colorbar(scatter, ax=ax); cbar.set_label('Success %', color='white')
cbar.ax.yaxis.set_tick_params(color='white'); plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
ax.set_xlabel('Reward Std Dev (instability)', **label_cfg); ax.set_ylabel('Mean Reward', **label_cfg)
ax.set_title('Stability vs Performance', color='white', fontsize=11)
ax.tick_params(colors='white'); [s.set_edgecolor('#30363D') for s in ax.spines.values()]

plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/models/dqn/dqn_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print('Plot saved.')

## 7 · Learning Curves (Best 3 Experiments)

In [ ]:
top3_ids = df.head(3)['exp_id'].tolist()
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_facecolor('#161B22'); fig.patch.set_facecolor('#0D1117')
colors_line = ['#2EA043', '#58A6FF', '#F0883E']

for i, exp_id in enumerate(top3_ids):
    log_file = f"{PROJECT_ROOT}/models/dqn/exp_{exp_id:02d}/evaluations.npz"
    try:
        data = np.load(log_file)
        steps = data['timesteps']
        means = data['results'].mean(axis=1)
        stds  = data['results'].std(axis=1)
        exp_name = next(e['name'] for e in EXPERIMENTS if e['id'] == exp_id)
        ax.fill_between(steps, means-stds, means+stds, alpha=0.15, color=colors_line[i])
        ax.plot(steps, means, color=colors_line[i], lw=2, label=f"Exp {exp_id}: {exp_name}")
    except FileNotFoundError:
        print(f'Log not found for exp {exp_id}')

ax.set_xlabel('Environment Steps', color='white'); ax.set_ylabel('Mean Episode Reward', color='white')
ax.set_title('DQN Learning Curves — Top 3 Experiments', color='white', fontsize=13)
ax.tick_params(colors='white'); ax.legend(facecolor='#1C2128', labelcolor='white')
[s.set_edgecolor('#30363D') for s in ax.spines.values()]
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.savefig(f'{PROJECT_ROOT}/models/dqn/dqn_learning_curves.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()

## 8 · Behavioural Discussion

### What we observed across the 10 DQN experiments:

**Learning Rate (Exps 1, 2, 3)**
High LR (5e-3) causes Q-value divergence — the loss spikes and reward collapses after initial gains. Low LR (1e-4) converges reliably but slowly. The baseline (1e-3) strikes the best balance, reaching stable reward within 80k steps.

**Replay Buffer Size (Exps 4, 5)**
Small buffer (5k) leads to correlated updates: the agent repeatedly learns from recent identical situations, causing it to forget earlier experiences (catastrophic forgetting). Large buffer (200k) significantly improves stability and final performance — critical for our 90-step episodes where skin dynamics are gradual.

**Discount Factor Gamma (Exps 6, 7)**
Low γ (0.85) produces a myopic agent that repeatedly reaches for quick-win actions (pills) and ignores long-term habits (diet, exercise). High γ (0.999) slows credit assignment across 90 steps — the agent struggles to attribute which early actions caused late-episode success.

**Exploration Schedule (Exp 8)**
Extended ε keeps the agent exploring longer, which is beneficial in our stochastic environment where action effects have noise. The agent discovers the synergy of combining skincare + diet more reliably.

**Network Architecture (Exp 9)**
Deeper network [256,256,256] improves capacity but requires more data. With 150k steps it slightly underperforms the baseline — it needs more training to utilise its capacity.

**Target Network + Large Batch (Exp 10)**
More frequent target updates (100 steps) with large batches (256) produces the most stable Q-estimates. This configuration shows the lowest variance across evaluation runs, making it the most reliable for deployment.

In [ ]:
# Save final summary
summary = df[['exp_id','name','mean_reward','std_reward','success_rate']].copy()
print('\n🏆 FINAL DQN RANKINGS')
print('='*65)
for _, row in summary.iterrows():
    medal = '🥇' if _ == 0 else '🥈' if _ == 1 else '🥉' if _ == 2 else '  '
    print(f"{medal} #{_+1:2d} {row['name']:<35} R={row['mean_reward']:+6.2f} ± {row['std_reward']:.2f}  ✅{row['success_rate']:.0f}%")
print('='*65)

In [ ]:
# Download best model
best_id = df.iloc[0]['exp_id']
print(f'Best experiment: Exp {best_id} — {df.iloc[0]["name"]}')
import shutil
shutil.copy(f'{PROJECT_ROOT}/models/dqn/exp_{best_id:02d}/best_model.zip',
            f'{PROJECT_ROOT}/models/dqn/best_model.zip')
print('Best model copied to models/dqn/best_model.zip')
from google.colab import files
files.download(f'{PROJECT_ROOT}/models/dqn/best_model.zip')